# Reto proyecto Machine Learning II
## Análisis supervisado y optimización de predicciones sobre Titanic

### 1. Importación de librerías

In [3]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


ModuleNotFoundError: No module named 'pandas'

### 2. Carga del dataset

In [ ]:

df = pd.read_csv('/workspace/dataset.csv')
df.head()


### 3. Exploración inicial

In [ ]:

df.info()


In [ ]:

df.isnull().sum()


In [ ]:

df.describe(include='all')


### 4. Visualización rápida

In [ ]:

plt.figure(figsize=(8,4))
sns.countplot(data=df, x='Survived')
plt.title('Distribución de la variable objetivo')
plt.show()


In [ ]:

plt.figure(figsize=(8,4))
sns.histplot(df['Age'], kde=True)
plt.title('Distribución de Age')
plt.show()


In [ ]:

plt.figure(figsize=(8,4))
sns.boxplot(x=df['Fare'])
plt.title('Boxplot de Fare')
plt.show()


### 5. Definición de variables
Se elimina `Survived` de las variables predictoras. También se elimina `PassengerId` porque es un identificador.

In [ ]:

X = df.drop(columns=['Survived', 'PassengerId'])
y = df['Survived']

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object']).columns.tolist()

print('Variables categóricas:', categorical_features)
print('Variables numéricas:', numeric_features)


### 6. División train/test

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


### 7. Preprocesamiento
- Imputación de nulos
- One-Hot Encoding para variables categóricas
- StandardScaler para variables numéricas

In [ ]:

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


### 8. Definición de los cinco modelos

In [ ]:

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'SVC': SVC(probability=True, kernel='rbf', C=1.0),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
}


### 9. Entrenamiento y evaluación de los modelos

In [ ]:

results = []
trained_pipelines = {}

for name, model in models.items():
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    clf.fit(X_train, y_train)
    trained_pipelines[name] = clf

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': report['1']['precision'],
        'Recall': report['1']['recall'],
        'F1-score': report['1']['f1-score'],
        'ROC AUC': roc_auc
    })

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
results_df


### 10. Matrices de confusión

In [ ]:

for name, clf in trained_pipelines.items():
    y_pred = clf.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de confusión - {name}')
    plt.xlabel('Predicción')
    plt.ylabel('Real')
    plt.show()


### 11. Reporte de clasificación

In [ ]:

for name, clf in trained_pipelines.items():
    y_pred = clf.predict(X_test)
    print(name)
    print(classification_report(y_test, y_pred, zero_division=0))
    print('-' * 60)


### 12. Curvas ROC

In [ ]:

plt.figure(figsize=(8,6))

for name, clf in trained_pipelines.items():
    y_prob = clf.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curvas ROC')
plt.legend()
plt.show()


### 13. Comparación final

In [ ]:

results_df


In [ ]:

best_model = results_df.iloc[0]
print('Mejor modelo según accuracy:')
print(best_model)


### 14. Conclusión

In [ ]:

print('Se ha realizado el preprocesamiento del dataset Titanic con imputación, One-Hot Encoding y escalado.')
print('Se han entrenado cinco modelos de clasificación: Regresión Logística, KNN, SVC, Árbol de Decisión y Random Forest.')
print('La comparación final permite identificar qué modelo ofrece el mejor rendimiento sobre el conjunto de prueba.')
